In [2]:
from pathlib import Path
import shutil
import sys

import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

PYTHON_SRC = Path("../../python/src").resolve()

if str(PYTHON_SRC) not in sys.path:
    sys.path.insert(0, str(PYTHON_SRC))

/Users/kashyaprajpurohit/myGitHubRepos/kr-ai-workshop/rag-platform/python/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from rag_platform.chunking.recursive import RecursiveChunker
from rag_platform.connectors.filesystem import FilesystemConnector
from rag_platform.domain.models import (
    DocumentSource,
    SourceType,
)
from rag_platform.domain.types import SourceID, TenantID
from rag_platform.parsers.markdown import MarkdownParser
from rag_platform.parsers.pdf import PDFParser
from rag_platform.parsers.text import TextParser

In [4]:
DATA_ROOT = Path( "../../data/raw/ZCompanyLLC" ).resolve()

source = DocumentSource(
    id=SourceID("zcompanyllc-hr"),
    tenant_id=TenantID("zcompanyllc"),
    source_type=SourceType.FILE,
    uri=str(DATA_ROOT),
)

documents = FilesystemConnector(source).read()

parsers = {
    "markdown": MarkdownParser(),
    "pdf": PDFParser(),
    "txt": TextParser(),
}

parsed_documents = [
    parsers[document.format.value].parse(document)
    for document in documents
]

chunker = RecursiveChunker(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = []

for document in parsed_documents:
    chunks.extend(chunker.chunk(document))

print(
    f"Documents: {len(documents)}, "
    f"Chunks: {len(chunks)}"
)

Documents: 85, Chunks: 88


In [5]:
MODEL_NAME = "BAAI/bge-small-en-v1.5"

model = SentenceTransformer(MODEL_NAME)

chunk_texts = [
    chunk.text
    for chunk in chunks
]

embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
)

print(embeddings.shape)

Batches: 100%|██████████| 3/3 [00:00<00:00, 16.52it/s]

(88, 384)


In [6]:
# create a persstent local chroma store at "data/processed/vector_store_expt/"
VECTOR_STORE_PATH = Path( "../../data/processed/vector_store_expt").resolve()

VECTOR_STORE_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

client = chromadb.PersistentClient(
    path=str(VECTOR_STORE_PATH)
)

print(VECTOR_STORE_PATH)

/Users/kashyaprajpurohit/myGitHubRepos/kr-ai-workshop/rag-platform/data/processed/vector_store_expt


In [7]:
# create a collection
collection = client.get_or_create_collection(
    name="zcompany_hr_chunks",
    metadata={
        "embedding_model": MODEL_NAME,
        "embedding_dimension": "384",
    },
)

print("Collection:", collection.name)
print("Count:", collection.count())

Collection: zcompany_hr_chunks
Count: 88


Insert chunks
ID - embedding - metadata 
The actual text can also be stored so the eperiment can retrive the chunk directly.

In [8]:
collection.add(
    ids=[str(chunk.chunk_id) for chunk in chunks],
    embeddings=embeddings.tolist(),
    documents=[chunk.text for chunk in chunks],
    metadatas=[
        {
            "document_id": str(chunk.document_id),
            "sequence_number": chunk.sequence_number,
            "chunking_version": chunk.chunking_version,
        }
        for chunk in chunks
    ],
)

print("Count:", collection.count())

Count: 88


Query the Vector Store 

In [9]:
query = (
    "How many days can an employee based in India "
    "work remotely each week?"
)

query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
)[0]

In [10]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5,
)

for rank, (
    chunk_id,
    document,
    distance,
) in enumerate(
    zip(
        results["ids"][0],
        results["documents"][0],
        results["distances"][0],
    ),
    start=1,
):
    print("=" * 80)
    print(f"Rank: {rank}")
    print(f"ID: {chunk_id}")
    print(f"Distance: {distance:.4f}")
    print(document[:500])

Rank: 1
ID: 92229214bff5812084d99491aa93721c95154cad86f46223b18f3a4666b76f3b
Distance: 0.3136
# India Work From Home Policy

Employees based in India may request up to 2 regular remote-working days per week.

Temporary work from another country:
- Must be recorded before travel.
- Requires manager approval.
- Germany- and India-based employees are normally subject to host-country leave/holiday rules while temporarily working abroad.
- Sweden-based employees remain under Swedish HQ policy globally.

Remote work does not itself change an employee's permanent base location.

Rank: 2
ID: 249509402ad2ba740e346b1fa0974d8e7f76cb7391a2186741b3b6805b81477a
Distance: 0.4230
# Germany Work From Home Policy

Employees based in Germany may request up to 2 regular remote-working days per week.

Temporary work from another country:
- Must be recorded before travel.
- Requires manager approval.
- Germany- and India-based employees are normally subject to host-country leave/holiday rules while temporar

Persistence test
Note: Restart your notebook kernel, or simply create a new client.

In [11]:
reopened_client = chromadb.PersistentClient(
    path=str(VECTOR_STORE_PATH)
)

reopened_collection = (
    reopened_client.get_collection(
        name="zcompany_hr_chunks"
    )
)

print(
    "Persisted count:",
    reopened_collection.count(),
)

Persisted count: 88


### Conclusion 

Current result
ChromaDB: 1.5.9
Documents: 85
Chunks: 88
Vectors: 88 × 384
Collection: zcompany_hr_chunks
Persisted count after reopening: 88 ✅
Top-1: India WFH Policy ✅

The distance ordering also matches our earlier cosine ranking:
Chroma distance     Cosine similarity
0.3136              0.8432
0.4230              0.7885
0.4384              0.7808
0.4937              0.7532
0.5993              0.7003

And there's a very simple relationship here: distance ≈ 1 - cosine_similarity

For example: 1 - 0.8432 = 0.1568 which is not 0.3136 

So, Chroma's distance is actually using a squared-distance formulation for this collection, which is consistent with: 2 × (1 - cosine)
For rank 1: 2 × (1 - 0.8432) = 0.3136

Therefore, ChromaDB 1.5.9 is selected as the prototype vector store.

The experiment verified:

- 88 BGE-small embeddings can be stored successfully.
- Chunk text and metadata can be stored alongside vectors.
- Similarity search returns the expected ranking.
- The persisted collection can be reopened successfully.
- Chroma's reported distance preserves the same ranking as our cosine
  similarity calculation.

For the prototype, ChromaDB provides sufficient local persistence and
vector-search capability without introducing an external database service.
